# Plato's Pizza : création de la base SQLite
Auteur : Zohair

## Sommaire
1. Lecture des CSV
2. Création des tables
3. Chargement des données
4. Contrôles

## 1. Lecture des CSV

In [2]:
import pandas as pd
import sqlite3                                     # SQLite est déjà inclus dans Python
orders = pd.read_csv("data/orders.csv")
details = pd.read_csv("data/order_details.csv")
pizzas = pd.read_csv("data/pizzas.csv")
types = pd.read_csv("data/pizza_types.csv", encoding="cp1252")   # encodage vu à l'exploration

## 2. Création des tables

In [3]:
con = sqlite3.connect("pizza.db")                                  # crée (ou ouvre) le fichier pizza.db
script = open("sql/00_creation_tables.sql", encoding="utf-8").read()   # lit le script SQL
con.executescript(script)                                          # exécute toutes les instructions du script

## 3. Chargement des données

In [4]:
types.to_sql("pizza_types", con, if_exists="append", index=False)    # remplit la table (append = garde la structure créée)
pizzas.to_sql("pizzas", con, if_exists="append", index=False)
orders.to_sql("orders", con, if_exists="append", index=False)
details.to_sql("order_details", con, if_exists="append", index=False)

48620

## 4. Contrôles

In [5]:
# même nombre de lignes que dans les CSV ?
requete = """
SELECT (SELECT COUNT(*) FROM orders) AS orders,
       (SELECT COUNT(*) FROM order_details) AS order_details,
       (SELECT COUNT(*) FROM pizzas) AS pizzas,
       (SELECT COUNT(*) FROM pizza_types) AS pizza_types
"""                              # compte les lignes de chaque table, sur une seule ligne de résultat
pd.read_sql(requete, con)        # envoie la requête à pizza.db et affiche le tableau

,orders,order_details,pizzas,pizza_types
0,21350,48620,96,32


### Observation
La base contient le même nombre de lignes que les CSV : orders 21350, order_details 48620, pizzas 96, pizza_types 32. Aucune ligne n'a été perdue au chargement.

In [6]:
# la jointure des 4 tables perd-elle des lignes ?
requete = """
SELECT COUNT(*) AS nb_lignes
FROM order_details d
JOIN orders o ON d.order_id = o.order_id
JOIN pizzas p ON d.pizza_id = p.pizza_id
JOIN pizza_types t ON p.pizza_type_id = t.pizza_type_id
"""                              # relie chaque ligne vendue à sa commande, sa pizza et son type
pd.read_sql(requete, con)

,nb_lignes
0,48620


In [7]:
# l’ingrédient avec l’apostrophe est-il bien enregistré ?
pd.read_sql("SELECT name, ingredients FROM pizza_types WHERE ingredients LIKE '%Nduja%'", con)   # LIKE '%...%' = contient

,name,ingredients
0,The Calabrese Pizza,"‘Nduja Salami, Pancetta, Tomatoes, Red Onions,..."


### Observations
- La jointure des 4 tables donne 48620 lignes, autant que order_details : aucune ligne ne se perd en reliant les tables.
- L'ingrédient ‘Nduja Salami (The Calabrese Pizza) est bien enregistré dans la base, avec son apostrophe.
- Conclusion : la base pizza.db est complète et prête pour les analyses.

In [8]:
con.close()   # ferme la connexion pour libérer pizza.db (sinon DBeaver peut le trouver verrouillé)